# Final Demo — Recursive LM for Poker

**STAT 4830 Final Project · Aadithya Srinivasan, Aryan Arora, Aarav M.**

This notebook reproduces the headline result end-to-end in Google Colab:

1. Clone the repo and install dependencies.
2. Load the final REINFORCE LoRA checkpoint (uploaded alongside this notebook).
3. Run three live poker scenarios with full retrieve → compute → decide reasoning traces.
4. Compare zero-shot vs BC vs RL vs heuristic on a 30-episode eval.

**Runtime:** ~10 minutes on a free-tier Colab T4 GPU.

**Setup:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

## 1. Install dependencies and clone the repo

In [ ]:
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes matplotlib

import os
if not os.path.exists('STAT-4830-RL-project'):
    !git clone https://github.com/aryanarora236/STAT-4830-RL-project.git
os.chdir('STAT-4830-RL-project')
!git pull --ff-only

import sys
sys.path.insert(0, '.')

print('Repo ready.')

In [ ]:
# Sanity check
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')

In [ ]:
# Verify tests pass (~2 seconds)
!python -m pytest tests/ -q --tb=line 2>&1 | tail -5

## 2. Download the RL checkpoint

The LoRA adapter from the final REINFORCE run lives in Google Drive. If your team has not uploaded it yet, you can still follow the rest of this notebook — it will fall back to evaluating the zero-shot + heuristic comparison only.

In [ ]:
# Replace this Drive URL with the RL checkpoint zip your team uploads.
# Expected contents after unzip: checkpoints/poker_rl/adapter_config.json, adapter_model.safetensors, tokenizer.*
RL_CHECKPOINT_URL = os.environ.get('RL_CHECKPOINT_URL', '')
HAS_RL_CKPT = False

if RL_CHECKPOINT_URL:
    import urllib.request
    print(f'Downloading RL checkpoint from {RL_CHECKPOINT_URL}...')
    urllib.request.urlretrieve(RL_CHECKPOINT_URL, 'poker_rl.zip')
    !unzip -q -o poker_rl.zip
    HAS_RL_CKPT = os.path.exists('checkpoints/poker_rl/adapter_config.json')
else:
    HAS_RL_CKPT = os.path.exists('checkpoints/poker_rl/adapter_config.json')

print(f'RL checkpoint present: {HAS_RL_CKPT}')

## 3. Load the model

We load the base Qwen2.5-Coder-1.5B-Instruct once and reuse it for zero-shot, BC, and RL by switching LoRA adapters.

In [ ]:
from src.training import load_model

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

if HAS_RL_CKPT:
    print('Loading RL checkpoint...')
    model, tokenizer = load_model('./checkpoints/poker_rl', load_in_4bit=True)
else:
    print('Loading base model (zero-shot evaluation only)...')
    model, tokenizer = load_model(MODEL_ID, load_in_4bit=True, lora_r=16)

model.eval()
print('Ready.')

## 4. Three live demo scenarios

For each scenario we:
1. Generate a poker task.
2. Show the first 600 characters of the context.
3. Ask the RL-trained model to produce code, then execute it.
4. Show the heuristic's answer for comparison.
5. Print whether the model matched.

In [ ]:
import random
from src.poker.tasks import generate_poker_task, generate_preflop_task, generate_postflop_task
from src.poker.agents import PokerLocalLLMAgent, PokerHeuristicAgent
from src.poker.rewards import parse_action

random.seed(4830)

llm_agent = PokerLocalLLMAgent(
    model=model, tokenizer=tokenizer,
    name='Demo-RL-Qwen1.5B', max_steps=5, temperature=0.1,
)
heuristic = PokerHeuristicAgent()

scenarios = [
    ('Preflop',  generate_preflop_task),
    ('Flop',     generate_postflop_task),
    ('All streets (random)', generate_poker_task),
]

for label, gen in scenarios:
    print('=' * 72)
    print(f'SCENARIO: {label}')
    print('=' * 72)
    context, question, correct = gen()
    print('CONTEXT (first 600 chars):')
    print(context[:600].rstrip() + ('...' if len(context) > 600 else ''))
    print()
    print('QUESTION:', question.split(chr(10))[0])
    print()

    print('--- LLM agent ---')
    llm_pred, llm_trace = llm_agent.run_episode(context, question, correct)
    print(f'Predicted: {llm_pred!r}  (steps: {len(llm_trace)})')
    # Print the last code block the model ran
    codes = [e.get('code', '') for e in llm_trace if 'code' in e]
    if codes:
        last_code = codes[-1]
        print('Last code (first 400 chars):')
        print(last_code[:400].rstrip())

    print()
    print('--- Heuristic ---')
    heur_pred, _ = heuristic.run_episode(context, question, correct)
    print(f'Predicted: {heur_pred!r}')
    print(f'Correct:   {correct!r}')

    match = parse_action(llm_pred)[0] == parse_action(correct)[0]
    print(f'LLM match: {"YES" if match else "NO"}')
    print()

## 5. Quantitative comparison — 30 episodes per street

Same task set for every agent, seed fixed.

In [ ]:
from src.poker.evaluation import PokerEvaluationFramework

random.seed(2026)
agents = [llm_agent, heuristic]

print('Running all-streets eval (30 episodes)...')
eval_all = PokerEvaluationFramework(agents=agents, task_generator=generate_poker_task, num_episodes=30)
eval_all.run_evaluation()
eval_all.display_results()
eval_all.display_confusion_matrix(llm_agent.name)

print()
print('Summary (from export_run_summary):')
summary = eval_all.export_run_summary()
for agent_name, metrics in summary['agents'].items():
    print(f'  {agent_name:30s}  type_match_rate={metrics["type_match_rate"]:.3f}  avg_reward={metrics["avg_reward"]:.3f}')

## 6. Compare against the published final eval

If `experiments/results/final_eval_*.json` is present in the repo, load and print it. These are the numbers we reported in `report.md` and the final presentation.

In [ ]:
import glob, json
candidates = sorted(glob.glob('experiments/results/final_eval_*.json'))
if not candidates:
    print('No final_eval_*.json found — this notebook is running against a freshly cloned repo before the team pushed final results.')
else:
    latest = candidates[-1]
    print(f'Latest final eval: {latest}')
    with open(latest) as f:
        payload = json.load(f)
    meta = payload.get('meta', {})
    print('Meta:', meta)
    print()
    for suite_name, suite in payload.get('suites', {}).items():
        print(f'--- {suite_name} ---')
        for agent, metrics in suite.get('agents', {}).items():
            print(f'  {agent:30s} type_match={metrics.get("type_match_rate", 0):.3f}  n={metrics.get("total_episodes", 0)}')
        print()

## Appendix — reproducing the full training pipeline

This notebook only runs inference. The full BC + REINFORCE pipeline requires an Ampere+ GPU (A100, L40, H100) and ~90 minutes. The exact commands are in `scripts/primeintellect/README.md`.

Quick reference:

```bash
prime pods create --id <H100-availability-id> --env HF_TOKEN=$HF_TOKEN --name stat4830
prime pods ssh <pod-id>

# On the pod:
INSTALL_UNSLOTH=1 bash <(curl -sSL https://raw.githubusercontent.com/aryanarora236/STAT-4830-RL-project/main/scripts/primeintellect/bootstrap.sh)
cd /root/STAT-4830-RL-project && source .venv/bin/activate
bash scripts/primeintellect/run_full.sh
```

End-to-end this produces:
- `checkpoints/poker_bc/` — BC LoRA adapters
- `checkpoints/poker_rl/` + `iter_*/` — RL checkpoints every 5 iters
- `checkpoints/poker_rl/training_curves.png` — reward + accuracy EMA
- `experiments/results/final_eval_*.json` — structured per-suite metrics

Fill the result tables in `report.md` / `README.md` / `self_critique_week15.md` / `docs/final_slides_outline.md` from the eval JSON with:

```bash
python scripts/fill_report_from_eval.py --eval experiments/results/final_eval_<TIMESTAMP>.json
```